<a href="https://colab.research.google.com/github/lakaderohan16-del/olist-revenue-analytics-tableau/blob/main/01_load_and_validate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
base = '/content/drive/MyDrive/olist-project'
os.makedirs(f"{base}/data/raw", exist_ok=True)
os.makedirs(f"{base}/data/clean", exist_ok=True)
os.makedirs(f"{base}/sql", exist_ok=True)

print("project folder exists:", os.path.exists(base))
print("raw folder exists:", os.path.exists(f"{base}/data/raw"))
print(os.listdir(f"{base}/data/raw"))


project folder exists: True
raw folder exists: True
[]


In [4]:
print(sorted(os.listdir(f"{base}/data/raw")))

['Copy of olist_customers_dataset.csv', 'Copy of olist_geolocation_dataset.csv', 'Copy of olist_order_items_dataset.csv', 'Copy of olist_order_payments_dataset.csv', 'Copy of olist_order_reviews_dataset.csv', 'Copy of olist_orders_dataset.csv', 'Copy of olist_products_dataset.csv', 'Copy of olist_sellers_dataset.csv', 'Copy of product_category_name_translation.csv']


In [5]:
import sqlite3, pandas as pd
raw = f"{base}/data/raw"

conn = sqlite3.connect(f"{base}/olist.db")
for f in sorted(os.listdir(raw)):
    if f.lower().endswith(".csv"):
        name = f.replace("_dataset", "").replace(".csv", "")
        df = pd.read_csv(f"{raw}/{f}")
        df.to_sql(name, conn, if_exists="replace", index=False)
        print(name, df.shape)

Copy of olist_customers (99441, 5)
Copy of olist_geolocation (1000163, 5)
Copy of olist_order_items (112650, 7)
Copy of olist_order_payments (103886, 5)
Copy of olist_order_reviews (99224, 7)
Copy of olist_orders (99441, 8)
Copy of olist_products (32951, 9)
Copy of olist_sellers (3095, 4)
Copy of product_category_name_translation (71, 2)


In [6]:
pd.read_sql("""
SELECT order_status, COUNT(*) AS orders
FROM olist_orders
GROUP BY order_status
ORDER BY orders DESC
""", conn)

DatabaseError: Execution failed on sql '
SELECT order_status, COUNT(*) AS orders
FROM olist_orders
GROUP BY order_status
ORDER BY orders DESC
': no such table: olist_orders

In [7]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)["name"].tolist()
for t in tables:
    new = t.replace("Copy of ", "").strip()
    if new != t:
        conn.execute(f'ALTER TABLE "{t}" RENAME TO {new}')
conn.commit()

print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

                                name
0                    olist_customers
1                  olist_geolocation
2                  olist_order_items
3               olist_order_payments
4                olist_order_reviews
5                       olist_orders
6                     olist_products
7                      olist_sellers
8  product_category_name_translation


In [8]:
pd.read_sql("""
SELECT order_status, COUNT(*) AS orders
FROM olist_orders
GROUP BY order_status
ORDER BY orders DESC
""", conn)

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [9]:
pd.read_sql("""
WITH monthly AS (
    SELECT substr(o.order_purchase_timestamp, 1, 7) AS month,
           ROUND(SUM(i.price), 2) AS gmv
    FROM olist_orders o
    JOIN olist_order_items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
)
SELECT month, gmv,
       ROUND(100.0 * (gmv - LAG(gmv) OVER (ORDER BY month)) / LAG(gmv) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly
ORDER BY month
""", conn)

,month,gmv,mom_growth_pct
0,2016-09,134.97,NaN
1,2016-10,40325.11,29777.1
2,2016-12,10.90,-100.0
3,2017-01,111798.36,1025573.0
4,2017-02,234223.40,109.5
5,2017-03,359198.85,53.4
6,2017-04,340669.68,-5.2
7,2017-05,489338.25,43.6
8,2017-06,421923.37,-13.8
9,2017-07,481604.52,14.1


In [10]:
pd.read_sql("""
SELECT substr(order_purchase_timestamp, 1, 7) AS month,
       COUNT(*) AS delivered_orders,
       ROUND(100.0 * AVG(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date THEN 1 ELSE 0 END), 1) AS on_time_pct
FROM olist_orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
GROUP BY month
ORDER BY month
""", conn)

,month,delivered_orders,on_time_pct
0,2016-09,1,0.0
1,2016-10,265,98.9
2,2016-12,1,100.0
3,2017-01,750,96.9
4,2017-02,1653,96.8
5,2017-03,2546,94.4
6,2017-04,2303,92.1
7,2017-05,3545,96.4
8,2017-06,3135,96.1
9,2017-07,3872,96.6


In [11]:
checks = {
 "duplicate order_ids in orders": "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM olist_orders",
 "delivered orders with no delivery date": "SELECT COUNT(*) FROM olist_orders WHERE order_status='delivered' AND order_delivered_customer_date IS NULL",
 "delivered before purchase (impossible)": "SELECT COUNT(*) FROM olist_orders WHERE order_delivered_customer_date < order_purchase_timestamp",
 "orders with no items": "SELECT COUNT(*) FROM olist_orders o LEFT JOIN olist_order_items i ON o.order_id=i.order_id WHERE i.order_id IS NULL",
 "items whose order is missing": "SELECT COUNT(*) FROM olist_order_items i LEFT JOIN olist_orders o ON i.order_id=o.order_id WHERE o.order_id IS NULL",
 "orders with more than one review": "SELECT COUNT(*) FROM (SELECT order_id FROM olist_order_reviews GROUP BY order_id HAVING COUNT(*)>1)",
 "distinct customer_id": "SELECT COUNT(DISTINCT customer_id) FROM olist_customers",
 "distinct customer_unique_id": "SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers",
}
results = [(name, conn.execute(q).fetchone()[0]) for name, q in checks.items()]
pd.DataFrame(results, columns=["check", "count"])

,check,count
0,duplicate order_ids in orders,0
1,delivered orders with no delivery date,8
2,delivered before purchase (impossible),0
3,orders with no items,775
4,items whose order is missing,0
5,orders with more than one review,547
6,distinct customer_id,99441
7,distinct customer_unique_id,96096
